# PROJECT 6 TEST
by Szymon Waliczek
 - Ballistic Andreev transport in 2DEG side Josephson junction - semi/super-conducting hybrid **InAs-SC**
 - System nr 1 Horizontal wire with 2 semi leads(up/down) and 1 SC lead on the right edge.
 - System nr 2 Horizontal wire with 2 semi leads(up/down) and  SC block on the right edge.
 - We analyze the relation of dispersion using wrapped system with Y translational symmetry
 - **With Peierls phase**
 - **No spin**
 - **With type electron pairing**

In [ ]:
import ipyparallel as ipp
#-------------------------------------------------------------------------------------------------------------------
cluster = ipp.Client(profile="kwant_parallel_test")
#-------------------------------------------------------------------------------------------------------------------
#from tera import TeraClient
#cluster = TeraClient(username="swaliczek", profile_name="slurm")
#-------------------------------------------------------------------------------------------------------------------
v = cluster[:]
lview = cluster.load_balanced_view()
len(v)

In [ ]:
%%px --local

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
import pickle
import adaptive
adaptive.notebook_extension()
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interactive, HBox, VBox, fixed
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

In [ ]:
%%px --local

import kwant
import tinyarray
import numpy as np
from time import perf_counter
from scipy.sparse.linalg import eigsh
#-------------------------------------------------------------------------------------------------------------------
class Timer():
    def __init__(self, name):
        self.name = name
        self.engine_id = os.environ.get('IPY_ENGINE_ID', '0')
    
    def __enter__(self):
        self.start = perf_counter()
        return self

    def __exit__(self, *args):
        self.end = perf_counter()
        if self.engine_id == '0':
            t = self.end - self.start
            print(f"Time - {self.name}: {t/60:.3f} min")
#-------------------------------------------------------------------------------------------------------------------
# Physical constants
from scipy.constants import physical_constants
eV = physical_constants['electron volt'][0]
m_el = physical_constants['electron mass'][0]
h_bar = physical_constants['Planck constant over 2 pi'][0]
phi_0 = physical_constants['elementary charge over h-bar'][0]
#-------------------------------------------------------------------------------------------------------------------
a = 10
m_eff = 0.023 * m_el
t = (h_bar**2 / (2 * m_eff * (a*1e-9)**2)) / eV # hopping in eV
freedom_deg = 2; # e + holes
PI = np.pi
#-------------------------------------------------------------------------------------------------------------------
tau_0 = tinyarray.array([[1, 0], [0, 1]])
tau_x = tinyarray.array([[0, 1], [1, 0]])
tau_y = tinyarray.array([[0, -1j], [1j, 0]])
tau_z = tinyarray.array([[1, 0], [0, -1]])

In [ ]:
%%px --local

def onsite(site, mu, B, delta, x0):
    return (4*t - mu) * tau_z
#-------------------------------------------------------------------------------------------------------------------
def onsite_sc(site, mu, B, delta, x0):
    return (4*t - mu)*tau_z + delta*tau_x
#-------------------------------------------------------------------------------------------------------------------
def hop(site1, site2, mu, B, delta, x0):
    x1, y1 = site1.pos
    x2, y2 = site2.pos
    if x1 < 0:
        phi = phi_0 * B * ((x1 + x2)/2 + x0) * (y1 - y2)*1e-18
        p_phase = tinyarray.array([[np.exp(-1j * phi), 0], 
                              [0, np.exp(1j * phi)]])
        return -t * tau_z * p_phase
    return -t * tau_z * 1.0 # A=0
#-------------------------------------------------------------------------------------------------------------------    
def make_systems(a, W, W_sc_block, L, t):
    # SC hybrid  
    lat = kwant.lattice.square(a, norbs=2)
    sys_hybrid = kwant.Builder()
    sys_hybrid[lat.shape(lambda pos: -W <= pos[0] < 0 and -L/2 <= pos[1] <= L/2, (-a, 0))] = onsite # Site N
    sys_hybrid[lat.shape(lambda pos: pos[0] == 0 and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_sc # Site SC
    sys_hybrid[lat.neighbors()] = hop
    lead_n = kwant.Builder(kwant.TranslationalSymmetry((0, -a)), conservation_law=-tau_z, particle_hole=tau_y) # Lead N
    lead_n[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
    lead_n[lat.neighbors()] = hop
    sys_hybrid.attach_lead(lead_n);
    sys_hybrid.attach_lead(lead_n.reversed());
    lead_sc = kwant.Builder(kwant.TranslationalSymmetry((a, 0))) # Lead SC
    lead_sc[lat.shape(lambda pos: -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_sc
    lead_sc[lat.neighbors()] = hop    
    sys_hybrid.attach_lead(lead_sc)
    #-------------------------------------------------------------------------------------------------------------------
    # SC block
    sys_block = kwant.Builder()
    sys_block[lat.shape(lambda pos: -W <= pos[0] < 0 and -L/2 <= pos[1] <= L/2, (-a, 0))] = onsite # Site N
    if(W_sc_block != 0): sys_block[lat.shape(lambda pos: 0 <= pos[0] <= W_sc_block and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_sc # Site SC
    sys_block[lat.neighbors()] = hop
    sys_block.attach_lead(lead_n)
    sys_block.attach_lead(lead_n.reversed())
    # Normal
    if(W_sc_block == 0): return sys_block.finalized()
    return sys_hybrid.finalized(), sys_block.finalized()
#-------------------------------------------------------------------------------------------------------------------
def make_wrapped_system(a, W, W_sc_wrap, L, t):
    # Wraparound
    lat = kwant.lattice.square(a, norbs=2)
    sys_wrap = kwant.Builder(kwant.TranslationalSymmetry((0, a)))
    sys_wrap[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
    if(W_sc_wrap != 0): sys_wrap[lat.shape(lambda pos: 0 <= pos[0] <= W_sc_wrap, (0, 0))] = onsite_sc
    sys_wrap[lat.neighbors()] = hop
    return kwant.wraparound.wraparound(sys_wrap, coordinate_names='y').finalized()
#-------------------------------------------------------------------------------------------------------------------
from functools import lru_cache
@lru_cache(maxsize=1)
def initialize_wrapped(_a, _W, _W_sc_wrap, _L, _t):
    ini_wrap = make_wrapped_system(a=_a, W=_W, W_sc_wrap=_W_sc_wrap, L=_L, t=_t)
    return ini_wrap
#-------------------------------------------------------------------------------------------------------------------
def compute_eigen(k, sys_wrap, modes, mu, B, delta, x0):
    params = dict(mu=mu, B=B, delta=delta, x0=x0, k_y=k)
    H = sys_wrap.hamiltonian_submatrix(params=params, sparse=True)
    evals, evecs = eigs(H, k=modes, sigma=0, which='LM')
    idx = evals.real.argsort()
    return evals[idx].real, evecs[:, idx]
#-------------------------------------------------------------------------------------------------------------------
def compute_Gj(sys, E, p, j):
    smatrix = kwant.smatrix(sys, E, params=p)
    if j == 0:
        n0 = smatrix.submatrix((0, 0), (0, 0)).shape[0]
        ree = smatrix.transmission((0, 0), (0, 0))
        rhe = smatrix.transmission((0, 1), (0, 0))
        return n0 - ree + rhe
    
    tee = smatrix.transmission((j, 0), (0, 0))
    the = smatrix.transmission((j, 1), (0, 0))
    return tee - the
#-------------------------------------------------------------------------------------------------------------------
def compute_eigen_energies(_k, _sys_wrap, _modes, _mu, _B, _delta, _x0):
    p = dict(mu=_mu, B=_B, delta=_delta, x0=_x0, k_y=_k)
    H = _sys_wrap.hamiltonian_submatrix(params=p, sparse=True)
    evals = eigsh(H, k=_modes, sigma=0, which='LM', return_eigenvectors=False)
    return np.sort(evals.real)
#-------------------------------------------------------------------------------------------------------------------
def compute_modes_lead(_sys, _delta, _x0, _mu, _b, _E):
    p = dict(mu=_mu, B=_b, delta=_delta, x0=_x0)
    prop_modes, _ = _sys.leads[0].modes(energy=_E, params=p)
    return len(prop_modes.momenta)

In [ ]:
def plot_sys(sys, x, y, name, p):
    with Timer(name):
        COLOR = lambda site: np.real(sys.hamiltonian(site, site, params = p)[0,1])
        kwant.plot(sys, fig_size=(x, y), show=False, site_color = COLOR)
        plt.title(f"{name} a = {a}nm")
        plt.xlabel("x [nm]")
        plt.ylabel("y [nm]")
        plt.show()
#-------------------------------------------------------------------------------------------------------------------
def eigen(a, W, W_sc_wrap, L, t, modes, xlim, nk, p):
    with Timer('eigen'):
        k_range = np.linspace(-xlim, xlim, nk)
        results = lview.map_sync(lambda k: compute_eigen(k, initialize_wrapped(a, W, W_sc_wrap, L, t), 
                                                         modes, p['mu'], p['B'], p['delta'], p['x0']), k_range)
        E = np.array([x[0] for x in results])
        V = np.array([x[1] for x in results])
    return E, k_range, V
#-------------------------------------------------------------------------------------------------------------------
def plot_bands_wrap(E, k_range, dk, name, L, s, xlim, ylim, p):
    with Timer(name):
        modes = E.shape[1]
        plt.figure(figsize=(s,s))
        for i in range(modes):
            plt.plot(k_range, E[:, i]*1e3, 'k.', markersize=1)
        info = (f"s-wave \n{name} \nL = {L} nm \n$\mu = {p['mu']*1e3}$ meV \n$B = {p['B']}$ T\
        \n$\Delta = {p['delta']*1e3}$ meV \n$\delta_k = {dk:.3f}$ 1/a")
        plt.text(1.05*xlim, 0, info, fontsize=3*s)
        plt.hlines(y=0, xmin=-dk/2, xmax=dk/2, colors='red', label='$\delta_k$', lw=1)
        plt.axhline(p['delta']*1e3, color='blue', label='$\Delta$', lw=0.2)
        plt.axhline(p['mu']*1e3, color='green', label='$\mu$', lw=0.3)
        plt.axhline(-p['delta']*1e3, color='blue', lw=0.2)
        plt.xlabel("$k_y$ [1/a]"); plt.ylabel("$E [meV]$")
        plt.xlim(-xlim, xlim); plt.ylim(-ylim, ylim)
        plt.yticks(np.arange(-ylim, ylim+ylim/10, ylim/5))
        plt.xticks(np.arange(-xlim, xlim+xlim/10, xlim/2))
        plt.legend(loc='upper right', fontsize=2*s); plt.grid(alpha=0.3); plt.show()
#-------------------------------------------------------------------------------------------------------------------
def get_dk(E, k_range):
    k_l, k_p = -t, t
    prev = a*t
    
    n_k = len(k_range)
    n_modes = E.shape[1]
    start = n_k//2 if n_k%2 == 0 else n_k//2+1 # we do not want the k=0 point since it is not possible unless delta==0?
    for i in range(start, n_k):
        curr_k = k_range[i]
        for m in range(n_modes-1):
            E1, E2 = Evals[i, m], Evals[i, m+1]
            if E1*E2 < 0 and abs(E2) < prev:
                prev = E2
                k_p = curr_k
    dk = 2 * k_p; k_l = -k_p
    print(f"k_l = {k_l:.5f}, k_p = {k_p:.5f} \ndelta_k = {dk:.5f}")
    return dk
#-------------------------------------------------------------------------------------------------------------------
def show_E_k(Evals, k_range):
    n_k = len(k_range)
    n_modes = Evals.shape[1]

    for i in range(n_k - 1):
        if abs(k_range[i]) < 0.4:
            print(f"\nk_{i} =  {k_range[i]}")
            for m in range(10):
                print(f"E_{i}_{m} = {Evals[i, m]*1e3}\t")
#-------------------------------------------------------------------------------------------------------------------
def compute_G_mu_B(sys, _E, _delta, _x0, _mu, _B, _j, A):
    def map_G(tuple):
        mu_val, B_val = tuple
        p = dict(mu=mu_val, B=-B_val, delta=_delta, x0=_x0)
        return compute_Gj(sys, E=_E, p=p, j=_j)
    learner = adaptive.Learner2D(map_G, bounds=[_mu, _B])
    runner = adaptive.Runner(learner, executor = cluster, goal=lambda l: l.loss() < A)
    runner.live_info()
    return learner, runner
#-------------------------------------------------------------------------------------------------------------------
def to_file(l, r, filename):
    with open(f"6_data/{filename}_data.pkl", "wb") as f: pickle.dump(l.data, f)
    t = r.elapsed_time()
    P = len(l.data)
    del l; del r
    return f"6_data/{filename}_data.pkl", t, P
#-------------------------------------------------------------------------------------------------------------------
def plot_G_mu_B(E, name, L, s, delta, x0, _mu, _B, t, P, N, v_map, filename):
    with Timer('plot_G_mu_B'):
        with open(filename, "rb") as f: data = pickle.load(f)
        tmp_learner = adaptive.Learner2D(lambda x: x, bounds=[_mu, _B])
        tmp_learner.data = data
        del data
        x, y, z = tmp_learner.interpolated_on_grid(n=N) # !!! z = [len(y), len(z)] !!!
        del tmp_learner
        v_abs = max(abs(z.min()), abs(z.max()))
        plt.figure(figsize=(1.5*s, s))
        im = plt.pcolormesh(x*1e3, y, z.T, cmap='seismic', vmin=-v_abs, vmax=v_abs)
        info = (f"s-wave \n{name} \nL = {L} nm \n$E = {E*1e3:.2f}$ meV\n$\Delta = {delta*1e3:.2f}$ meV \n$t = {t/60:.2f}min$ \n$p = {P}|{N}$")
        plt.text(1.3, 0.5, info, transform=plt.gca().transAxes, fontsize=3*s, verticalalignment='center')
        plt.colorbar(im, label='$G$ [$e^2/h$]')
        plt.xlabel('$\mu$ [meV]'); plt.ylabel('$-B$ [T]')
        plt.title(f'G($\mu, -B, E={E:.2f}$ meV)')
        plt.tight_layout()
#-------------------------------------------------------------------------------------------------------------------        
        mu_vec = np.linspace(_mu[0], _mu[1], v_map.shape[0]) * 1e3
        B_vec = np.linspace(_B[0], _B[1], v_map.shape[1])
        levels = [1, 5, 9, 13]
        cs = plt.contour(mu_vec, B_vec, v_map.T, levels=levels, colors='black', linewidths=0.6)
        plt.clabel(cs, inline=True, fontsize=2*s, fmt={1:'2', 5:'4', 9:'6', 13:'8'})
#-------------------------------------------------------------------------------------------------------------------
        plt.savefig(f"6_plots/{name}_{P}_{N}_plot.png", dpi=150, bbox_inches='tight')
        plt.close()
#-------------------------------------------------------------------------------------------------------------------
def get_modes_wrapped(_a, _W, _W_sc_wrap, _L, _t, _delta, _x0, _mu, _B, N, nk, modes, xlim, tol):
    with Timer('modes_wrapped'):
        k_range = np.linspace(0, xlim, nk)
        mu_vec = np.linspace(_mu[0], _mu[1], N)
        B_vec = np.linspace(_B[0], _B[1], N)
        map_v = np.zeros((N, N), dtype=int)

        for i in range(N):
            for j in range(N):
                count = 0
                mu_i = mu_vec[i]
                B_i = B_vec[j]
                spectrum_at_k = lview.map_sync(lambda k: compute_eigen_energies(k, initialize_wrapped(_a, _W, _W_sc_wrap, _L, _t),
                                                                        modes, mu_i, B_i, _delta, _x0), k_range)
                spectrum_at_k = np.array(spectrum_at_k)
                for idx in range(spectrum_at_k.shape[1]):
                    if np.any(abs(spectrum_at_k[:, idx]) < tol): count += 2
                map_v[i, j] = count
    return map_v
#-------------------------------------------------------------------------------------------------------------------
def get_modes_lead(sys, delta, x0, mu, B, N, E):
    with Timer('modes_lead'):
        mu_vec = np.linspace(mu[0], mu[1], N)
        B_vec = np.linspace(B[0], B[1], N)
        points = [(m, b) for m in mu_vec for b in B_vec]
        results = lview.map_sync(lambda p: compute_modes_lead(sys, delta, x0, p[0], p[1], E), points)
        map_v = np.array(results).reshape(N, N)
    return map_v

In [ ]:
%%px --local

L200 = 200
W100 = 100
Wscwrap100 = 100
Wscblock100 = 100
freedom_deg = 2; # e + holes

In [ ]:
p_test = dict(mu=0, B=0, delta=0.001, x0=0, k_y=0)

In [ ]:
h_100, b_100 = make_systems(a, W100, Wscblock100, L, t)
w_100 = make_wrapped_system(a, W100, Wscwrap100, L, t)
#-------------------------------------------------------------------------------------------------------------------
plot_sys(h_100, 2, 3, 'HYBRID_100', p_test)
plot_sys(w_100, 2.7, 2, 'WRAP_100', p_test)

In [ ]:
params_edge = dict(mu=0.002, B=-0.3, delta=0.001, x0=0)

In [ ]:
Evals, k, Vecs = eigen(a, W100, Wscwrap100, L, t, modes=20, xlim=PI/4, nk=201, p=params_edge) # over 50 modes out of ylim
dk = get_dk(Evals, k)                                                               # for nk=101+n*100 except 701 ???
plot_bands_wrap(E=Evals, k_range=k, dk=dk, name='WRAP_100', L=L, s=4, xlim=PI/4, ylim=5, p=params_edge)

In [ ]:
G_map1 , R1 = compute_G_mu_B(sys=h_100, _E=0.0, _delta=0.001, _x0=0, _mu=(0, 0.01), _B=(0, 1.5), _j=1, A=0.005)

In [ ]:
f1, time1, points1 = to_file(l=G_map1, r=R1, filename='h_100_005')

In [ ]:
V_map_l = get_modes_lead(sys=h_100, delta=0.001, x0=0.0, mu=(0, 0.01), B=(0, 1.5), N=40, E=0)

In [ ]:
V_map_w =get_modes_wrapped(_a=a, _W=W100, _W_sc_wrap=Wscwrap100, _L=L200, _t=t, _delta=0.001, _x0=0.0, _mu=(0, 0.01), _B=(0, 1.5), 
                                N=40, nk=101, modes=10, xlim=PI, tol=0.2*1e-4)

In [ ]:
plot_G_mu_B(E=0.0, name='h_100_01', L=L, s=5, delta=0.001, x0=0, _mu=(0, 0.01), _B=(0, 1.5), t=time1, P=points1, N=500, filename=f1)

In [ ]:
map_wrapped = 